In [ ]:
import json
import time
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# LOAD Labeled_Data_for_Exclusion_Included

In [ ]:
# Paths
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/Screening_2/output_review.xlsx"
# Load existing processed file if it exists

df = pd.read_excel(SAVE_PATH)


In [ ]:
df.response.value_counts()

,count
response,
excluded,15695
included,236


In [ ]:
filter = df[df['response']=='included']

In [ ]:
# filter.sample(15).to_excel('/content/drive/MyDrive/fix_systematic_review/Screening_2/excluded_15.xlsx')

# Checking if it is a review study

In [ ]:
prompt = """
    "You will receive a title and an abstract. "
    "Your task is to determine whether the study is a review type "
    "(e.g., systematic review, scoping review, meta-analysis), "
    "an Opinion paper, or a Practice Implementation Report.\n\n"
    "If it matches one of these categories, answer 'Yes'. "
    "If not, answer 'No'.\n\n"
    "Title: {title}\n"
    "Abstract: {abstract}\n\n

    If the article is clearly an **empirical research study** in one of the following categories, it must be classified as:
    Final decision: No
    Rating: 1
    These research types are:
    • Qualitative research – Non-numerical data collected systematically (e.g., interviews, focus groups, thematic analysis).
    • Mixed methods – Integrates both qualitative and quantitative data and analyses.
    • Randomized controlled trial (RCT) – Random allocation, experimental manipulation, numerical outcome data, causal inference.
    • Quantitative non-randomized – Numerical assessment of effects without randomization (e.g., quasi-experimental/comparison groups).
    • Quantitative descriptive – Numerical data describing trends or characteristics, no manipulation, using descriptive statistics, and no qualitative analysis.

    "Decision: (Yes/No)\n"
    "Explain: (Brief explanation of why you decided this way)"
    """

In [ ]:

title = """
Overcoming challenges for nurse-led research in the neonatal intensive care unit
"""
abstract = """
Nurse researchers have made considerable contributions to the science of nursing. Conducting studies in the clinical environment is the best way to determine what interventions are most effective. Because most research is conducted by nurses from academic settings, they often are not viewed as credible by nurses in clinical practice, nor are they accepted readily into the neonatal intensive care unit (NICU) to conduct research. This article addresses the barriers to implementing and conducting studies in the NICU setting, and provides suggestions for creating collaborative and mutually satisfying relationships between nurse researchers and healthcare staff in the NICU. Partnerships built on mutual respect are necessary to have successful research outcomes, and to continue to build knowledge for the best neonatal care.
"""
full_prompt = prompt.format(
      title=title,
      abstract=abstract
  )


In [ ]:
import json
import time
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# Initialize OpenAI client
client = OpenAI(api_key="Api_key")

# Save every X seconds
SAVE_INTERVAL = 120
df = filter.copy()
# # Paths
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/Screening_2/output_review_systematic_1.xlsx"
INPUT_PATH = "/content/drive/MyDrive/fix_systematic_review/Screening_2/output_review.xlsx"


# Load existing processed file if it exists
if os.path.exists(SAVE_PATH):
    print(f"🔄 Loading existing file from: {SAVE_PATH}")
    df = pd.read_excel(SAVE_PATH)
else:
    print(f"📥 No existing output file found. Loading input file from: {INPUT_PATH}")
    df = pd.read_excel(INPUT_PATH)
    df = df[df['response']=='included']
    # Create 'response' column if not present
    if 'response_systematic' not in df.columns:
        df['response_systematic'] = None

# Track time for periodic save
last_save_time = time.time()

# Iterate with progress bar
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['response_systematic']):
        continue

    # Format the prompt
    full_prompt = prompt.format(
        title=row['Title'],
        abstract=row['Abstract']
    )

    try:
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            max_tokens=2048,
            top_p=0.45,
            messages=[
                {"role": "user", "content": full_prompt}
            ]
        )

        # Extract and store response
        content = response.choices[0].message.content
        print(content)
        df.at[index, 'response_systematic'] = content

    except Exception as e:
        print(f"❌ Error at index {index}: {e}")
        continue

    # Save periodically
    current_time = time.time()
    if current_time - last_save_time > SAVE_INTERVAL:
        df.to_excel(SAVE_PATH, index=False)
        print(f"💾 Checkpoint saved at {SAVE_PATH}")
        last_save_time = current_time

# Final save
df.to_excel(SAVE_PATH, index=False)
print(f"✅ Final save completed at {SAVE_PATH}")


📥 No existing output file found. Loading input file from: /content/drive/MyDrive/fix_systematic_review/Screening_2/output_review.xlsx


Processing abstracts:   0%|          | 1/236 [00:02<08:14,  2.10s/it]

Decision: No

Explain: The study does not fit into the categories of a review type, opinion paper, or practice implementation report. Instead, it appears to be an empirical research study that discusses barriers to research utilization in nursing, presenting findings from existing research rather than conducting a systematic review or providing an opinion. The focus on encouraging nurses to embrace evidence-based practice and the mention of conceptual models suggests it is more of a discussion or analysis rather than a formal review or opinion piece.


Processing abstracts:   1%|          | 2/236 [00:05<12:10,  3.12s/it]

Decision: No

Explain: The study described in the title and abstract focuses on the role of nursing faculty mentors in facilitating evidence-based practice among clinical nurses. It highlights the context, challenges, and successes of this mentorship initiative rather than systematically reviewing existing literature or presenting a meta-analysis. Therefore, it does not fit the criteria for a review type, opinion paper, or practice implementation report, but rather appears to be an empirical research study focused on a specific intervention in a healthcare setting.


Processing abstracts:   1%|▏         | 3/236 [00:09<12:38,  3.25s/it]

... (463 outputs left out)

Processing abstracts: 100%|█████████▉| 235/236 [07:20<00:01,  1.88s/it]

Decision: No

Explain: The study described in the title and abstract is an empirical research study that focuses on identifying barriers to research utilization among nurses. It employs a descriptive quantitative approach, utilizing a questionnaire and descriptive statistics for data analysis. This does not fit the categories of a review type (such as systematic review, scoping review, or meta-analysis), an opinion paper, or a practice implementation report. Instead, it is classified as a quantitative descriptive study, which is distinct from the specified categories.


Processing abstracts: 100%|██████████| 236/236 [07:22<00:00,  1.87s/it]

Decision: No

Explain: The study described in the title and abstract is an empirical research study that identifies barriers to research utilization in nursing practice through a descriptive methodology. It involves the collection and analysis of numerical data from a sample of nurses, which classifies it as a quantitative descriptive study rather than a review type, opinion paper, or practice implementation report. Therefore, it does not fit the specified categories for classification.
✅ Final save completed at /content/drive/MyDrive/fix_systematic_review/Screening_2/output_review_systematic_1.xlsx


In [ ]:
df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,response,response_systematic
8,12470013.0,10.1016/s0964339702000125,Barriers to research utilisation: the clinical...,2002,The advance of the evidence-based practice (EB...,[],en,included,Decision: No\n\nExplain: The study does not fi...
11,18838895.0,10.1097/01.NND.0000320688.57308.f9,Nursing faculty mentors as facilitators for ev...,2008,Increasing use of evidence-based practice (EBP...,[],en,included,Decision: No\n\nExplain: The study described i...
12,18710376.0,10.1111/j.1365-2702.2007.02040.x,The spectrum of barriers to and facilitators o...,2008,The focus of the study is the identification o...,[],en,included,Decision: No \nExplain: The study is an empir...
17,15761309.0,10.1097/00005110-200503000-00005,Overcoming barriers and promoting the use of r...,2005,To identify changes in nurse attitudes toward ...,[],en,included,Decision: No\n\nExplain: The study described i...
22,9362653.0,10.1097/00002800-199709000-00009,Barriers and facilitators to the utilization o...,1997,The nurse in clinical practice must demonstrat...,[],en,included,Decision: No\n\nExplain: The study described i...
...,...,...,...,...,...,...,...,...,...
15918,NaN,NaN,Communication and utilization of nursing resea...,1976,Explorative study was an attempt to identify s...,NaN,en,included,Decision: No\n\nExplain: The study described i...
15919,NaN,NaN,Analysis of the free questions of the BARRIERS...,2024,Aim: Examine and analyze the responses to free...,Nursing Research; Clinical Nursing Research; N...,en,included,Decision: No\n\nExplain: The study described i...
15921,NaN,NaN,BARRIERS TO NURSING RESEARCH: A COMPARATIVE ST...,2014,Nurses need to adapt to new models of care bas...,NURSING RESEARCH; COMPREHENSIVE HEALTH CARE; P...,en,included,Decision: No\n\nExplain: The study described i...
15922,NaN,NaN,Barriers to research utilization among nurses ...,2011,Aim: The study examines the nurses working at ...,Research utilization; nursing research; barrie...,en,included,Decision: No\n\nExplain: The study described i...


In [5]:
import pandas as pd
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/Screening_2/output_review_systematic_1.xlsx"
df = pd.read_excel(SAVE_PATH)

In [6]:
df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,response,response_systematic
0,12470013.0,10.1016/s0964339702000125,Barriers to research utilisation: the clinical...,2002,The advance of the evidence-based practice (EB...,[],en,included,Decision: No\n\nExplain: The study does not fi...
1,18838895.0,10.1097/01.NND.0000320688.57308.f9,Nursing faculty mentors as facilitators for ev...,2008,Increasing use of evidence-based practice (EBP...,[],en,included,Decision: No\n\nExplain: The study described i...
2,18710376.0,10.1111/j.1365-2702.2007.02040.x,The spectrum of barriers to and facilitators o...,2008,The focus of the study is the identification o...,[],en,included,Decision: No \nExplain: The study is an empir...
3,15761309.0,10.1097/00005110-200503000-00005,Overcoming barriers and promoting the use of r...,2005,To identify changes in nurse attitudes toward ...,[],en,included,Decision: No\n\nExplain: The study described i...
4,9362653.0,10.1097/00002800-199709000-00009,Barriers and facilitators to the utilization o...,1997,The nurse in clinical practice must demonstrat...,[],en,included,Decision: No\n\nExplain: The study described i...
...,...,...,...,...,...,...,...,...,...
231,NaN,NaN,Communication and utilization of nursing resea...,1976,Explorative study was an attempt to identify s...,NaN,en,included,Decision: No\n\nExplain: The study described i...
232,NaN,NaN,Analysis of the free questions of the BARRIERS...,2024,Aim: Examine and analyze the responses to free...,Nursing Research; Clinical Nursing Research; N...,en,included,Decision: No\n\nExplain: The study described i...
233,NaN,NaN,BARRIERS TO NURSING RESEARCH: A COMPARATIVE ST...,2014,Nurses need to adapt to new models of care bas...,NURSING RESEARCH; COMPREHENSIVE HEALTH CARE; P...,en,included,Decision: No\n\nExplain: The study described i...
234,NaN,NaN,Barriers to research utilization among nurses ...,2011,Aim: The study examines the nurses working at ...,Research utilization; nursing research; barrie...,en,included,Decision: No\n\nExplain: The study described i...


In [ ]:
import re
df["Extracted_decision_sytematic"] = df["response_systematic"].str.extract(
    r'(?:\*\*Decision:\*\*|Decision:)\s*(Yes|No)',
    flags=re.I
)


In [ ]:
df.Extracted_decision_sytematic.value_counts()

,count
Extracted_decision_sytematic,
No,208
Yes,28


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/Screening_2/output_review_systematic_label.xlsx"
df.to_excel(SAVE_PATH, index=False)

In [ ]:
df[df['Extracted_decision_sytematic']=='Yes']

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,response,response_systematic,Extracted_decision_sytematic
501,1.714776e+07,10.1111/j.1741-6787.2004.04003.x,Strategic collaborative model for evidence-bas...,2004,To describe a model that has been developed to...,[],en,included,Decision: Yes\n\nExplain: The study describes ...,Yes
1574,1.268060e+07,10.1097/00001786-200304000-00004,Research dissemination and utilization improvi...,2003,Research improves nursing and patient care out...,[],en,included,Decision: Yes \nExplain: The abstract indicat...,Yes
1587,1.832499e+07,10.1111/j.1365-2834.2007.00791.x,Increasing research evidence in practice: a po...,2008,To determine the extent to which clinical nurs...,[],en,included,Decision: Yes\n\nExplain: The study is a revie...,Yes
2411,1.459598e+07,10.1590/s0034-71672003000100012,[Evidence-based practice: strategies for its i...,2003,Evidence-based practice is an approach that en...,[],en,included,Decision: Yes \nExplain: The article discusse...,Yes
3208,1.903629e+07,10.1054/nepr.2002.0058,Research-based practice: myth or reality? A re...,2002,A research-practice gap still exists despite t...,[],en,included,Decision: Yes\n\nExplain: The title and abstra...,Yes
4564,3.255879e+07,10.1097/01.NURSE.0000668460.98211.39,Identifying international barriers and facilit...,2020,To define research utilization (RU) and identi...,[],en,included,Decision: Yes \nExplain: The study is classif...,Yes
4565,2.645709e+07,10.4103/1735-9066.164501,Barriers and facilitators of nursing research ...,2015,According to professionalization in nursing pr...,"['Barriers and facilitators', 'Iran', 'nursing...",en,included,Decision: Yes \nExplain: The study is a syste...,Yes
4587,2.122341e+07,10.1111/j.1365-2834.2010.01144.x,Evidence-based practice barriers and facilitat...,2011,The purpose of the present study is to examine...,[],en,included,Decision: Yes\n\nExplain: The study is classif...,Yes
4622,3.343430e+07,10.1111/jocn.15657,Interventions and strategies aimed at clinical...,2021,To review interventions and strategies designe...,"['Nursing', 'clinical academic', 'leadership',...",en,included,Decision: Yes\n\nExplain: The study is classif...,Yes
4623,3.393191e+07,10.1111/jocn.15830,Approaches for establishing and sustaining cli...,2022,To discuss the need for a formalised structure...,"['clinician-researcher', 'evidence-based pract...",en,included,Decision: Yes \nExplain: The study is classif...,Yes
